In [1]:
%%bash
set -e
mkdir -p /kaggle/temp
cd /kaggle/temp && rm -rf FreeFine && git clone -q https://github.com/CIawevy/FreeFine.git
python3 - <<'PY'
import pathlib
root = pathlib.Path("/kaggle/temp/FreeFine/evaluation/metrics")
p = root/"main.py"; p.write_text(p.read_text().replace("args.3d", "getattr(args, '3d')"))
for f in [root/"MD"/"mean_distance.py", root/"MD"/"dift_sd.py"]:
    f.write_text(f.read_text().replace("stabilityai/stable-diffusion-2-1",
                                       "sd2-community/stable-diffusion-2-1"))
print("patched main.py + MD SD-2.1 mirror")
PY

patched main.py + MD SD-2.1 mirror


Cell 1 — clone FreeFine + apply the 3 patches you used


In [2]:
%%bash
set -e
cd /kaggle/temp && rm -rf FreeFine && git clone -q https://github.com/CIawevy/FreeFine.git
python3 - <<'PY'
import pathlib
root = pathlib.Path("/kaggle/temp/FreeFine/evaluation/metrics")
p = root/"main.py"; p.write_text(p.read_text().replace("args.3d", "getattr(args, '3d')"))
for f in [root/"MD"/"mean_distance.py", root/"MD"/"dift_sd.py"]:
    f.write_text(f.read_text().replace("stabilityai/stable-diffusion-2-1",
                                       "sd2-community/stable-diffusion-2-1"))
print("patched main.py + MD SD-2.1 mirror")
PY

patched main.py + MD SD-2.1 mirror


Cell 2 — build metric_env (torch 2.6/cu124 + your pinned fixes)


In [3]:
%%bash
set -e
pip install -q --root-user-action=ignore uv
uv python install 3.10.13
VENV=/kaggle/temp/metric_env; PY=$VENV/bin/python; REPO=/kaggle/temp/FreeFine
rm -rf $VENV && uv venv --python 3.10.13 $VENV
uv pip install --python $PY torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu124
uv pip install --python $PY "setuptools<70" wheel pip
grep -vi '^clip' $REPO/evaluation/metrics/requirements.txt > /tmp/metric_req.txt
uv pip install --python $PY -r /tmp/metric_req.txt
uv pip install --python $PY "setuptools<70"                 # re-pin before clip build
uv pip install --python $PY --no-build-isolation "clip @ git+https://github.com/openai/CLIP.git@dcba3cb2e2827b402d2701e7e1c7d9fed8a20ef1"
uv pip install --python $PY "pyarrow<16"                    # datasets/ImageReward compat
wget -q https://dl.fbaipublicfiles.com/mmf/clip/bpe_simple_vocab_16e6.txt.gz -P /tmp
for d in $(find $VENV -path '*/site-packages/clip' -o -path '*open_clip' -type d); do cp /tmp/bpe_simple_vocab_16e6.txt.gz "$d/" 2>/dev/null || true; done
echo "metric_env ready"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 75.2 MB/s eta 0:00:00
metric_env ready


 Downloaded cpython-3.10.13-linux-x86_64-gnu (download)
Installed Python 3.10.13 in 1.57s
 + cpython-3.10.13-linux-x86_64-gnu (python3.10)
Using CPython 3.10.13
Creating virtual environment at: /kaggle/temp/metric_env
Activate with: source /kaggle/temp/metric_env/bin/activate
Using Python 3.10.13 environment at: /kaggle/temp/metric_env
Resolved 27 packages in 839ms
 Downloaded nvidia-cuda-cupti-cu12
 Downloaded nvidia-cuda-nvrtc-cu12
 Downloaded torchaudio
 Downloaded torchvision
 Downloaded nvidia-nvjitlink-cu12
 Downloaded pillow
 Downloaded networkx
 Downloaded numpy
 Downloaded triton
 Downloaded nvidia-curand-cu12
 Downloaded nvidia-cusolver-cu12
 Downloaded nvidia-cusparselt-cu12
 Downloaded sympy
 Downloaded nvidia-nccl-cu12
 Downloaded nvidia-cufft-cu12
 Downloaded nvidia-cusparse-cu12
 Downloaded nvidia-cublas-cu12
 Downloaded nvidia-cudnn-cu12
 Downloaded torch
Prepared 27 packages in 48.20s
Installed 27 packages in 275ms
 + filelock==3.29.0
 + fsspec==2026.4.0
 + jinja2==3.1

Cell 3 — download GeoBench-2D (source images + masks + annotation)


In [4]:
import os
from huggingface_hub import snapshot_download
import os
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"  # avoid extra dep; keep simple
snapshot_download(repo_id="CIawevy/GeoBenchMeta", repo_type="dataset",
    local_dir="/kaggle/temp/GeoBenchMeta", token=os.environ.get("HF_TOKEN"),
    max_workers=4, etag_timeout=30,
    allow_patterns=["annotation_2d.json", "Geo-Bench-2D/**"])
print("GeoBench-2D downloaded")

Fetching ... files: 0it [00:00, ?it/s]

GeoBench-2D downloaded


Cell 4 — restore your 5677 generated PNGs + manifest into GeoBenchMeta


In [5]:
import os, glob, json, shutil
DS  = "/kaggle/input/datasets/georgiostzamouranis/freefine-geobench2d-bggen"
GEO = "/kaggle/temp/GeoBenchMeta"
PNG_ROOT = os.path.join(DS, "gen_results_2d_final", "gen_results_2d_backup")
assert os.path.isdir(PNG_ROOT), f"not found: {PNG_ROOT} — fix DS"

# Expose the read-only mounted PNGs at the path the manifest expects (instant, no copy)
link = os.path.join(GEO, "Gen_results_FreeFine_2d")
if os.path.islink(link):   os.remove(link)
elif os.path.isdir(link):  shutil.rmtree(link)
os.symlink(PNG_ROOT, link)
print("symlinked", link, "->", PNG_ROOT)
print("PNGs visible:", len(glob.glob(f"{GEO}/Gen_results_FreeFine_2d/**/*.png", recursive=True)))

# Reconstruct manifest from annotation_2d.json (identical logic to the Og run)
man = f"{GEO}/generated_results_freefine_2d.json"
ann = json.load(open(f"{GEO}/annotation_2d.json")); added = 0
for d, da in ann.items():
    for i, ins in da.get("instances", {}).items():
        for e in list(ins):
            rel = f"Gen_results_FreeFine_2d/{d}/{i}/{e}.png"
            if os.path.exists(os.path.join(GEO, rel)):
                ins[e]["gen_img_path"] = rel; added += 1
json.dump(ann, open(man, "w"))
print("manifest reconstructed, entries:", added)   # expect 5677

symlinked /kaggle/temp/GeoBenchMeta/Gen_results_FreeFine_2d -> /kaggle/input/datasets/georgiostzamouranis/freefine-geobench2d-bggen/gen_results_2d_final/gen_results_2d_backup
PNGs visible: 5677
manifest reconstructed, entries: 5677


Cell 5 — MD across BOTH GPUs (the parallel run)


In [6]:
import os, sys, re, json, time, select, subprocess
GEO="/kaggle/temp/GeoBenchMeta"; FREEFINE="/kaggle/temp/FreeFine"
METRIC_PY="/kaggle/temp/metric_env/bin/python"; MD=f"{FREEFINE}/evaluation/metrics"
os.environ.setdefault("HF_HOME","/kaggle/temp/hf")
from huggingface_hub import snapshot_download                      # warm SD-2.1 once
snapshot_download("sd2-community/stable-diffusion-2-1", token=os.environ.get("HF_TOKEN"),
    allow_patterns=["*.json","*.txt","tokenizer/*","scheduler/*","feature_extractor/*",
                    "text_encoder/*.bin","unet/*.bin","vae/*.bin"]); print("SD-2.1 cached",flush=True)

data=json.load(open(f"{GEO}/generated_results_freefine_2d.json"))
leaves=[(d,i,c) for d,da in data.items() for i,ins in da["instances"].items() for c in ins]
mid=len(leaves)//2; subs=[set(leaves[:mid]),set(leaves[mid:])]
print(f"{len(leaves)} cases -> {len(subs[0])}/{len(subs[1])}",flush=True)
def build(s):
    o={}
    for d,da in data.items():
        ni={i:{c:v for c,v in ins.items() if (d,i,c) in s} for i,ins in da["instances"].items()}
        ni={i:k for i,k in ni.items() if k}
        if ni: nd={k:v for k,v in da.items() if k!="instances"}; nd["instances"]=ni; o[d]=nd
    return o
half=[f"/kaggle/temp/md_half_{i}.json" for i in (0,1)]
for i,s in enumerate(subs): json.dump(build(s),open(half[i],"w"))
def cmd(p): return [METRIC_PY,"main.py","--path",p,"--use_relative_path","--base_dir",GEO,
    "--fid_path",f"{GEO}/Geo-Bench-2D/source_img_full_v2","--task","000000100","--level","0"]
base=os.environ.copy(); base.update({"PYTORCH_CUDA_ALLOC_CONF":"expandable_segments:True",
    "MPLBACKEND":"Agg","PYTHONUNBUFFERED":"1","HF_HOME":"/kaggle/temp/hf"})
procs,bufs=[],["",""]
for g in (0,1):
    e=base.copy(); e["CUDA_VISIBLE_DEVICES"]=str(g)
    procs.append(subprocess.Popen(cmd(half[g]),stdout=subprocess.PIPE,stderr=subprocess.STDOUT,env=e,cwd=MD,bufsize=0))
print("MD launched on GPU0 + GPU1\n",flush=True)

# ---- live progress: % done, s/it, ETA per GPU + combined, every 15s ----
HB=15
fds={p.stdout.fileno():i for i,p in enumerate(procs)}; openf=set(fds); start=last=time.time()
done={0:False,1:False}
def stat(b):
    m=re.findall(r"(\d+)/(\d+)\s*\[([^\]]*)\]", b[-6000:])
    return (int(m[-1][0]),int(m[-1][1]),m[-1][2]) if m else (None,None,None)
def line():
    parts=[]; cc=tt=0
    for i in range(2):
        if done[i]: parts.append(f"GPU{i}: DONE"); continue
        c,t,info=stat(bufs[i])
        if c is None:
            ls=[l for l in bufs[i][-1500:].replace("\r","\n").splitlines() if l.strip()]
            parts.append(f"GPU{i}: {(ls[-1][:42] if ls else 'starting...')}")
        else:
            cc+=c; tt+=t; parts.append(f"GPU{i}: {c}/{t} {100*c//max(t,1)}% [{info}]")
    comb=f"  ||  total {100*cc//tt}% ({cc}/{tt})" if tt else ""
    el=int(time.time()-start)
    return f"[t={el//60}m{el%60:02d}s] " + "  ||  ".join(parts) + comb
while openf:
    for fd in select.select(list(openf),[],[],1.0)[0]:
        ch=os.read(fd,65536)
        if not ch:
            i=fds[fd]; openf.discard(fd); done[i]=True
            print(f"   >>> GPU{i} finished (exit {procs[i].poll()})",flush=True); continue
        bufs[fds[fd]]+=ch.decode("utf-8","replace")
    if time.time()-last>=HB or not openf:
        print(line(),flush=True); last=time.time()
for p in procs: p.wait()

def parse(b):
    md=re.findall(r"MD:\s*([-\d.eE]+)",b); tot=[int(x) for x in re.findall(r"/(\d+)\s*\[",b)]
    return (float(md[-1]) if md else None),(max(tot) if tot else None),len(re.findall(r"Error in get_Matches",b))
print("\n===== combine =====",flush=True); sw=0.0; nt=0
for i,p in enumerate(procs):
    md,tot,err=parse(bufs[i])
    if md is None or tot is None: print(f"[GPU{i}] PARSE FAIL exit {p.returncode}\n"+bufs[i][-1200:]); continue
    c=tot-err; print(f"[GPU{i}] MD={md:.6f} over {c} cases (pairs {tot}, skipped {err})"); sw+=md*c; nt+=c
if nt: print(f"\nCOMBINED MD = {sw/nt:.6f} over {nt} cases")

Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

SD-2.1 cached
5677 cases -> 2838/2839
MD launched on GPU0 + GPU1

[t=0m15s] GPU0: starting...  ||  GPU1: starting...
[t=0m30s] GPU0: starting...  ||  GPU1: starting...
[t=0m45s] GPU0: -----MD-----  ||  GPU1: -----MD-----
[t=1m01s] GPU0: 6/6 100% [00:00<00:00,  7.10it/s]  ||  GPU1: 6/6 100% [00:00<00:00,  7.09it/s]  ||  total 100% (12/12)
[t=1m16s] GPU0: 1/2838 0% [00:13<10:42:08, 13.58s/it]  ||  GPU1: 1/2839 0% [00:13<10:45:17, 13.64s/it]  ||  total 0% (2/5677)
[t=1m32s] GPU0: 3/2838 0% [00:28<6:51:53,  8.72s/it]  ||  GPU1: 3/2839 0% [00:28<6:52:19,  8.72s/it]  ||  total 0% (6/5677)
[t=1m47s] GPU0: 5/2838 0% [00:42<6:05:18,  7.74s/it]  ||  GPU1: 5/2839 0% [00:42<6:02:27,  7.67s/it]  ||  total 0% (10/5677)
[t=2m02s] GPU0: 7/2838 0% [00:57<5:53:08,  7.48s/it]  ||  GPU1: 7/2839 0% [00:56<5:47:24,  7.36s/it]  ||  total 0% (14/5677)
[t=2m18s] GPU0: 9/2838 0% [01:11<5:46:01,  7.34s/it]  ||  GPU1: 9/2839 0% [01:11<5:45:28,  7.32s/it]  ||  total 0% (18/5677)
[t=2m33s] GPU0: 11/2838 0% [01:26<5

Cell 6 — FID_DINO + FID_KD (fast, run after MD finishes)


In [7]:
%%bash
set -e
source /kaggle/temp/metric_env/bin/activate
export MPLBACKEND=Agg HF_HOME=/kaggle/temp/hf TORCH_HOME=/kaggle/temp/torch
cd /kaggle/temp/FreeFine/evaluation/metrics
for T in 000000010 000000001; do
  python main.py --path /kaggle/temp/GeoBenchMeta/generated_results_freefine_2d.json \
    --use_relative_path --base_dir /kaggle/temp/GeoBenchMeta \
    --fid_path /kaggle/temp/GeoBenchMeta/Geo-Bench-2D/source_img_full_v2 --task $T --level 0
done

-----FID_DINO-----
FID_DINO: 487.82308893222034
-----Result-----
FID_DINO: 487.82308893222034
-----FID_KD-----
FID_KD: 0.1422931578762432
-----Result-----
FID_KD: 0.1422931578762432


Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /kaggle/temp/torch/hub/main.zip
Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vitb14/dinov2_vitb14_pretrain.pth" to /kaggle/temp/torch/hub/checkpoints/dinov2_vitb14_pretrain.pth
100%|██████████| 330M/330M [00:01<00:00, 268MB/s]
100%|██████████| 89/89 [01:40<00:00,  1.12s/it]
Using cache found in /kaggle/temp/torch/hub/facebookresearch_dinov2_main
MMD: 100%|██████████| 100/100 [00:09<00:00, 10.06it/s, mean=0.142]
